### GUV automatic analysis script (full data saving + quality check version)

In [ ]:
# in this *_53version, try to use percentile rather than mean to calculate the background,
# since another GUV drifting into a corner or a piece of debris will significantly interfere
# with the calculation when simply using 4-corner sampling and the mean


In [ ]:
%matplotlib inline  # display plots directly in Jupyter Notebook
import cv2  # OpenCV for image processing (e.g., drawing circular masks)
import numpy as np  # numerical computing, array operations
import pandas as pd  # data handling and saving to CSV
import pims  # for reading biological image sequences (e.g., TIF stacks)
import os  # file paths and directories
import glob  # pattern matching for file names (e.g., all .tif files)
from matplotlib import pyplot as plt  # plotting

# Set global plotting style
plt.rcParams['figure.figsize'] = [10, 6]  # default figure size


In [ ]:
class GUVAnalyzer:
    def __init__(self, movie_path, centers_path, vesicle_id, folder='./', correction=[0, 0]):
        self.movie_path = os.path.join(folder, movie_path)
        self.movie_name = os.path.basename(movie_path)
        self.vesicle_id = vesicle_id
        self.folder = folder 
        
        self.df_centers = pd.read_csv(os.path.join(folder, centers_path))
        self.all_frames = pims.TiffStack_pil(self.movie_path)
        
        # Split channels
        self.ch0 = self.all_frames[slice(0, len(self.all_frames), 2)]  # membrane signal
        self.ch1 = self.all_frames[slice(1, len(self.all_frames), 2)]  # cargo/other signal
        
        target = self.df_centers[self.df_centers['id'] == vesicle_id]
        if target.empty:
            raise ValueError(f"Vesicle ID {vesicle_id} not found")
            
        x_raw = target['x'].values[0]
        y_raw = target['y'].values[0]
        self.center0px = np.array([x_raw + correction[0], y_raw + correction[1]]).astype(int)
        
        self.circles_per_frame = []
        self.results_df = None

    def _get_profiles(self, center, img, margin):
        """Get 1D intensity profiles across the vesicle center in x and y directions."""
        h, w = img.shape
        x_start, x_end = max(0, center[0]-margin), min(w, center[0]+margin)
        y_start, y_end = max(0, center[1]-margin), min(h, center[1]+margin)
        x_arr = np.average(img[:, x_start:x_end], axis=1)
        y_arr = np.average(img[y_start:y_end, :], axis=0)
        return x_arr, y_arr

    def _calc_threshold(self, center, img, margin=5, corner_size=30):
        """Core improvement: use the four image corners to estimate background with a percentile."""
        h, w = img.shape
        # Inner sampling: central (around vesicle) region, 5x5 area
        inside = np.average(img[max(0,center[1]-margin):min(h,center[1]+margin), 
                                max(0,center[0]-margin):min(w,center[0]+margin)])
        
        # External background: four corners, each corner_size x corner_size
        c1 = img[0:corner_size, 0:corner_size]                # top-left
        c2 = img[0:corner_size, w-corner_size:w]              # top-right
        c3 = img[h-corner_size:h, 0:corner_size]              # bottom-left
        c4 = img[h-corner_size:h, w-corner_size:w]            # bottom-right
        all_bg_pixels = np.concatenate([c1, c2, c3, c4])
        # Key optimization: use the 30th percentile instead of the mean
        # Rationale: even if 2 corners are bright (e.g., debris or another vesicle),
        # as long as some portion of the corners contains true background,
        # a low percentile will capture the darker background level.
        background = np.percentile(all_bg_pixels, 30)

        if inside > background * 1.5:
            thresh = 2 * inside
        else:
            thresh = inside + (background - inside) / 2
        return thresh, inside, background

    def _compute_circle(self, img, thresh_val, current_center, avg_size=15):
        """Estimate vesicle circle (center and radius) based on thresholded profiles."""
        x_arr, y_arr = self._get_profiles(current_center, img, avg_size) 

        # Vertical profile along x_arr (image rows)
        left_search = x_arr[0:current_center[1]] 
        x_idx_l = np.nonzero(left_search > thresh_val)[0][-1] if np.any(left_search > thresh_val) else 0 
        right_search = x_arr[current_center[1]:]
        x_idx_r = current_center[1] + (np.nonzero(right_search > thresh_val)[0][0] 
                                       if np.any(right_search > thresh_val) else len(right_search))
        
        # Horizontal profile along y_arr (image columns)
        left_y_search = y_arr[0:current_center[0]]
        y_idx_l = np.nonzero(left_y_search > thresh_val)[0][-1] if np.any(left_y_search > thresh_val) else 0
        right_y_search = y_arr[current_center[0]:]
        y_idx_r = current_center[0] + (np.nonzero(right_y_search > thresh_val)[0][0] 
                                       if np.any(right_y_search > thresh_val) else len(right_y_search))
        
        x_dist, y_dist = (x_idx_r - x_idx_l), (y_idx_r - y_idx_l)
        cx, cy = y_idx_l + int(y_dist/2), x_idx_l + int(x_dist/2)
        r = 1.1 * max(x_dist, y_dist) / 2  # slightly inflate radius
        return cx, cy, r

    def run_tracking(self):
        """Track vesicle position and size frame-by-frame on channel 1."""
        active_center = self.center0px
        for frame in self.ch1:
            t_val, _, _ = self._calc_threshold(active_center, frame)
            try:
                cx, cy, r = self._compute_circle(frame, t_val, active_center)
                self.circles_per_frame.append([cx, cy, r])
                active_center = np.array([cx, cy]).astype(int)
            except Exception:
                # If tracking fails for this frame, record None
                self.circles_per_frame.append(None)
    
    def save_diagnostics(self):
        """Visualization: save images showing 4-corner background regions and center sampling."""
        import matplotlib.patches as patches
        diag_dir = os.path.join(self.folder, f"DIAG_{os.path.splitext(self.movie_name)[0]}_V{self.vesicle_id}")
        if not os.path.exists(diag_dir):
            os.makedirs(diag_dir)
            
        for i, frame in enumerate(self.ch1):
            circ = self.circles_per_frame[i]
            h, w = frame.shape
            cs = 30  # corner_size
            
            fig, ax = plt.subplots(1, 2, figsize=(15, 7))
            ax[0].imshow(frame, cmap='gray')
            ax[0].set_title(f"Frame {i} - Tracking & 4-corner background")
            
            # Draw four red dashed rectangles at corners for background regions
            corners = [(0, 0), (w-cs, 0), (0, h-cs), (w-cs, h-cs)]
            for coord in corners:
                ax[0].add_patch(
                    patches.Rectangle(
                        coord, cs, cs, linewidth=2,
                        edgecolor='red', facecolor='none',
                        linestyle='--', zorder=10
                    )
                )

            if circ is not None:
                cx, cy, r = circ
                # Draw inner sampling region (blue rectangle)
                im = 5  # inner_margin
                ax[0].add_patch(
                    patches.Rectangle(
                        (cx-im, cy-im), im*2, im*2,
                        linewidth=2, edgecolor='cyan',
                        facecolor='none', zorder=10
                    )
                )
                # Draw tracking circle (green) and center point (red)
                ax[0].add_patch(plt.Circle((cx, cy), r, color='lime', fill=False, linewidth=2))
                ax[0].scatter([cx], [cy], color='red', s=20)

                # Right panel: thresholded mask for verification
                t_val, in_val, bg_val = self._calc_threshold([int(cx), int(cy)], frame)
                ax[1].imshow(frame > t_val, cmap='viridis')
                ax[1].set_title(f"Thresh: {t_val:.1f} (In: {in_val:.1f}, BG: {bg_val:.1f})")
            
            plt.tight_layout()
            plt.savefig(os.path.join(diag_dir, f"check_f{i:04d}.png"))
            plt.close(fig)  # explicitly close current figure
        plt.close('all')   # extra safety: clear any remaining figures

    def calculate_intensities(self, circle_thickness=12, corner_size=30):
        """Core improvement: use 4-corner percentile background for final intensity calculation."""
        data = []
        cs = corner_size
        for i in range(len(self.ch0)):
            circ = self.circles_per_frame[i]
            f0, f1 = self.ch0[i], self.ch1[i]
            # Height and width are identical for both channels
            h, w = f0.shape
        
            # --- Key improvement: use np.percentile(30) instead of np.mean ---
            # Extract pixels from 4 corners and combine them
            def get_robust_bg(img):
                c1 = img[0:cs, 0:cs].flatten()
                c2 = img[0:cs, w-cs:w].flatten()
                c3 = img[h-cs:h, 0:cs].flatten()
                c4 = img[h-cs:h, w-cs:w].flatten()
                all_pixels = np.concatenate([c1, c2, c3, c4])
                return np.percentile(all_pixels, 30)

            bg0 = get_robust_bg(f0)
            bg1 = get_robust_bg(f1)
            
            if circ is not None:
                cx, cy, r = circ
                mask = np.zeros(f0.shape, dtype=np.uint8)
                cv2.circle(mask, (int(cx), int(cy)), int(r), 255, circle_thickness)
                coords = np.where(mask == 255) 
                int0 = np.round(np.mean(f0[coords]), 2) if len(coords[0]) > 0 else 0
                int1 = np.round(np.mean(f1[coords]), 2) if len(coords[0]) > 0 else 0
                data.append([i, self.vesicle_id, self.movie_name, int0, int1, cx, cy, r, bg0, bg1])
            else:
                data.append([i, self.vesicle_id, self.movie_name, None, None, None, None, None, bg0, bg1])
                
        cols = ['FrameId', 'VesicleId', 'MovieName', 'IntCh0', 'IntCh1', 'xCoord', 'yCoord', 'radius', 'bg0', 'bg1']
        df = pd.DataFrame(data, columns=cols)
        df['Net0'] = df['IntCh0'] - df['bg0']
        df['Net1'] = df['IntCh1'] - df['bg1']
        self.results_df = df
        return df


In [ ]:
INPUT_FOLDER = './Split_By_Python_2Channel_Slices/'  # input folder
# INPUT_FOLDER = './test/'   
movie_files = glob.glob(os.path.join(INPUT_FOLDER, "*.tif"))  # find all .tif files
all_results = [] 

# Create folder for per-vesicle CSV results
result_csv_dir = os.path.join(INPUT_FOLDER, "Individual_CSVs")
if not os.path.exists(result_csv_dir):
    os.makedirs(result_csv_dir)

# Loop over each movie
for movie_path in movie_files:
    movie_name = os.path.basename(movie_path)
    centers_path = os.path.splitext(movie_name)[0] + ".csv"
    if not os.path.exists(os.path.join(INPUT_FOLDER, centers_path)):
        continue

    df_centers = pd.read_csv(os.path.join(INPUT_FOLDER, centers_path))
    # New: drop rows with empty id
    df_centers = df_centers.dropna(subset=['id'])   
    # Loop over each vesicle in this movie
    for v_id in df_centers['id'].unique():
        try:
            guv = GUVAnalyzer(movie_name, centers_path, v_id, folder=INPUT_FOLDER)
            guv.run_tracking()  # 1. run tracking
            
            # 1. Save diagnostic images for quality checking
            guv.save_diagnostics() 
            
            # 2. Calculate intensities and normalize
            df = guv.calculate_intensities(circle_thickness=12)
            f0_net0 = df.loc[0, 'Net0']
            f0_net1 = df.loc[0, 'Net1']
            df['Norm0'] = df['Net0'] / f0_net0 if f0_net0 != 0 else 0
            df['Norm1'] = df['Net1'] / f0_net1 if f0_net1 != 0 else 0
            
            # 3. Save full CSV for this vesicle (including x, y, r, bg, norm, etc.)
            csv_name = f"Data_{os.path.splitext(movie_name)[0]}_V{v_id}.csv"
            df.to_csv(os.path.join(result_csv_dir, csv_name), index=False)
            
            all_results.append(df)
            print(f"Done: {movie_name} - ID {v_id}")
            
            # Manually delete guv object to free memory
            del guv

        except Exception as e:
            print(f"Error ID {v_id}: {e}")

# 4. Save concatenated master table
master_df = pd.concat(all_results, ignore_index=True)
master_df.to_csv(os.path.join(INPUT_FOLDER, "Master_Results_All_Vesicles.csv"), index=False)


In [ ]:
# Summary statistics across vesicles for each frame
stats = master_df.groupby('FrameId').agg({
    'Norm0': ['mean', 'sem', 'count'],
    'Norm1': ['mean', 'sem', 'count']
})
stats.columns = ['_'.join(col).strip() for col in stats.columns.values]
stats.to_csv(os.path.join(INPUT_FOLDER, "Summary_Statistics.csv"))

# Plot aggregated normalized intensities (F/F0) with SEM shading
plt.figure(figsize=(10, 6))
for ch, color, label in [('Norm0', 'blue', 'Ch0 Membrane'), ('Norm1', 'red', 'Ch1 Cargo')]:
    plt.plot(stats.index, stats[f'{ch}_mean'], color=color, label=f'{label} (Avg)', lw=2)
    plt.fill_between(stats.index,
                     stats[f'{ch}_mean'] - stats[f'{ch}_sem'], 
                     stats[f'{ch}_mean'] + stats[f'{ch}_sem'],
                     color=color, alpha=0.2)

plt.axhline(1.0, color='black', linestyle='--', alpha=0.5)
plt.title(f'Aggregated Results (N={int(stats["Norm0_count"].max())} Vesicles)')
plt.xlabel('Frame')
plt.ylabel('Normalized Intensity (F/F0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(INPUT_FOLDER, "Final_Aggregated_Plot.png"))
plt.show()
